# 📚 RAG Knowledge Extraction System — Week 4
## NLP Analysis (Topic Modeling & Named Entity Recognition)

**Parallax Lab — RAG Pipeline Project**
**Intern:** Chashman Aslam
**Program:** Parallax Lab Internship

Week 4 builds directly on Week 2's chunked, embedded corpus and Week 3's generation layer.
This notebook enriches every chunk with **structured NLP metadata** and pushes that metadata
into ChromaDB so retrieval can be filtered, not just ranked by similarity.

Deliverables covered end-to-end:

1. **Topic modeling** (BERTopic, with a sklearn-LDA fallback) to discover corpus themes.
2. **Named Entity Recognition** (chosen over sentiment analysis — see Section 5 for why),
   evaluated for accuracy against a small hand-labeled gold set.
3. **Manual validation of topics** and explicit handling of edge cases (short documents, jargon).
4. **Integration of topic + entity metadata into ChromaDB** for filtered semantic retrieval.
5. **Documentation** of the effectiveness and accuracy of the extracted NLP metadata.



---
## 1. Install Required Libraries

New this week: `bertopic` (+ its `umap-learn`/`hdbscan` dependencies) for topic modeling, and
`spacy` with the small English model for named entity recognition. The spaCy model is
installed directly from its GitHub release wheel, which `pip` handles the same as any other
package URL.


In [1]:
import sys
import subprocess

INLINE_REQUIREMENTS = [
    "pandas>=2.0.0", "numpy>=1.24.0", "tqdm>=4.65.0", "matplotlib>=3.7.0",
    "scikit-learn>=1.3.0", "pyarrow>=13.0.0",
    "sentence-transformers>=2.6.0", "chromadb>=0.5.0",
    "bertopic>=0.16.0", "spacy>=3.7.0",
]
SPACY_MODEL_URL = (
    "https://github.com/explosion/spacy-models/releases/download/"
    "en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
)

print("Installing Week 4 packages (this includes BERTopic's dependencies -- can take a few minutes) ...")
cmd = [sys.executable, "-m", "pip", "install", "-q"] + INLINE_REQUIREMENTS + [SPACY_MODEL_URL]
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print(result.stderr[-3000:])
    raise RuntimeError("pip install failed -- see stderr above.")
print("\nAll packages installed (or already satisfied).")


Installing Week 4 packages (this includes BERTopic's dependencies -- can take a few minutes) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## 2. Load the Chunked Corpus + Embeddings + ChromaDB Collection

We reload exactly what Week 2 produced: `data/chunks.parquet` (one row per chunk, with
`doc_id`, `title`, `category`), `data/embeddings.npy` (the matching embedding matrix, so topic
modeling doesn't need to re-embed anything), and the persistent ChromaDB collection Weeks 2–3
already ingested into.

**Offline-safe fallback:** if Week 2's real output isn't present, we regenerate the same small
schema-matching synthetic corpus used as a fallback in Weeks 2–3, re-embed it with the same
offline-safe backend, and rebuild the collection -- so this notebook is never blocked by an
earlier week not having been (re-)run.


In [2]:
import re
import time
from pathlib import Path
from typing import List, Optional, Dict

import numpy as np
import pandas as pd
import chromadb
from tqdm.auto import tqdm

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "arxiv_chunks"


def build_fallback_corpus() -> pd.DataFrame:
    '''Same offline-safe synthetic stand-in used in Weeks 2-3, so this notebook still runs
    end-to-end if earlier weeks' real outputs aren't present.'''
    topics_seed = {
        "cs.CL": ("Transformer Language Model", "This paper introduces a transformer-based "
                  "architecture for natural language understanding. Self-attention mechanisms "
                  "allow the model to capture long-range dependencies between tokens far more "
                  "effectively than recurrent networks, evaluated on text classification and "
                  "question answering benchmarks."),
        "cs.CV": ("Convolutional Image Classification", "We propose a convolutional neural "
                  "network architecture for large-scale image classification. The network uses "
                  "residual connections to enable training of very deep architectures without "
                  "vanishing gradients, achieving strong accuracy on standard benchmarks."),
        "cs.LG": ("Gradient Based Optimization", "This work analyzes the convergence properties "
                  "of stochastic gradient descent under non-convex loss landscapes typical of "
                  "deep neural networks, deriving new bounds as a function of batch size and "
                  "learning rate schedule."),
        "cs.IR": ("Dense Retrieval Semantic Search", "We present a dense retrieval system that "
                  "encodes queries and documents into a shared embedding space using a "
                  "bi-encoder trained with contrastive loss, outperforming BM25 on open-domain "
                  "question answering benchmarks."),
        "cs.RO": ("Robot Manipulation Reinforcement Learning", "This paper studies reinforcement "
                  "learning for robotic manipulation tasks in cluttered environments, training a "
                  "policy on simulated and real-world data that generalizes to novel object "
                  "configurations not seen during training."),
    }
    rows = []
    for cat, (title_seed, para) in topics_seed.items():
        for i in range(15):
            text = (para + " ") * 2
            rows.append({
                "chunk_id": f"synthetic.{cat}.{i:04d}::chunk0",
                "doc_id": f"synthetic.{cat}.{i:04d}",
                "chunk_index": 0, "text": text.strip(), "char_count": len(text),
                "title": f"{title_seed} — Study {i + 1}", "category": cat,
                "published": f"2024-0{(i % 9) + 1}-15T00:00:00Z",
            })
    return pd.DataFrame(rows)


using_fallback = False
if (DATA_DIR / "chunks.parquet").exists():
    chunks_df = pd.read_parquet(DATA_DIR / "chunks.parquet")
    print(f"Loaded {len(chunks_df):,} chunks from Week 2.")
else:
    using_fallback = True
    chunks_df = build_fallback_corpus()
    chunks_df.to_parquet(DATA_DIR / "chunks.parquet", index=False)
    print(f"[fallback] data/chunks.parquet not found -- generated a {len(chunks_df)}-chunk "
          f"synthetic stand-in corpus (same schema as Week 2) so this notebook still runs "
          f"end-to-end. Re-run Weeks 1-2 for the real corpus.")

# --- Embedding backend (mirrors Week 2/3's offline-safe pattern) ---
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
embedding_backend, embed_texts = None, None
try:
    from sentence_transformers import SentenceTransformer
    _model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    embedding_backend = f"sentence-transformers ({EMBEDDING_MODEL_NAME})"

    def embed_texts(texts, show_progress=False):
        return np.asarray(_model.encode(list(texts), show_progress_bar=show_progress,
                                          normalize_embeddings=True))
    _ = embed_texts(["smoke test"])
except Exception as e:
    print(f"[warn] Could not load '{EMBEDDING_MODEL_NAME}' ({type(e).__name__}). "
          f"Falling back to offline TF-IDF + SVD.")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize as sk_normalize

    _vectorizer = TfidfVectorizer(max_features=20000, stop_words="english")
    _tfidf = _vectorizer.fit_transform(chunks_df["text"].tolist())
    _n_components = min(384, _tfidf.shape[1] - 1, chunks_df.shape[0] - 1)
    _svd = TruncatedSVD(n_components=max(_n_components, 2), random_state=42)
    _svd.fit(_tfidf)
    embedding_backend = "TF-IDF + TruncatedSVD (offline fallback)"

    def embed_texts(texts, show_progress=False):
        return sk_normalize(_svd.transform(_vectorizer.transform(list(texts))))

# --- Load or (re)compute the embedding matrix aligned to chunks_df's row order ---
if (DATA_DIR / "embeddings.npy").exists() and not using_fallback:
    embeddings = np.load(DATA_DIR / "embeddings.npy")
    if embeddings.shape[0] != len(chunks_df):
        print("[warn] embeddings.npy row count doesn't match chunks_df -- re-embedding.")
        embeddings = embed_texts(chunks_df["text"].tolist(), show_progress=True)
else:
    embeddings = embed_texts(chunks_df["text"].tolist(), show_progress=True)
    np.save(DATA_DIR / "embeddings.npy", embeddings)

print(f"Embedding backend: {embedding_backend} | matrix shape: {embeddings.shape}")

# --- Reconnect to (or build) the persistent ChromaDB collection ---
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
    name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
)
if collection.count() != len(chunks_df):
    print(f"[info] Collection has {collection.count():,} vectors but corpus has "
          f"{len(chunks_df):,} chunks -- (re)ingesting to bring them in sync.")
    metas = [{"doc_id": r.doc_id, "chunk_index": int(r.chunk_index), "title": r.title,
              "category": r.category, "published": str(r.published)}
             for r in chunks_df.itertuples(index=False)]
    for start in range(0, len(chunks_df), 500):
        end = start + 500
        collection.upsert(
            ids=chunks_df["chunk_id"].tolist()[start:end],
            embeddings=embeddings[start:end].tolist(),
            documents=chunks_df["text"].tolist()[start:end],
            metadatas=metas[start:end],
        )
print(f"ChromaDB ready: collection '{COLLECTION_NAME}' holds {collection.count():,} vectors.")


[fallback] data/chunks.parquet not found -- generated a 75-chunk synthetic stand-in corpus (same schema as Week 2) so this notebook still runs end-to-end. Re-run Weeks 1-2 for the real corpus.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding backend: sentence-transformers (all-MiniLM-L6-v2) | matrix shape: (75, 384)
[info] Collection has 0 vectors but corpus has 75 chunks -- (re)ingesting to bring them in sync.
ChromaDB ready: collection 'arxiv_chunks' holds 75 vectors.


---
## 3. Topic Modeling — BERTopic (with an LDA Fallback)

**Approach:** BERTopic clusters the *same* embeddings already computed in Week 2 (passed in
directly, so it doesn't re-embed anything or need a fresh model download), reduces them with
UMAP, clusters with HDBSCAN, and extracts a c-TF-IDF keyword signature per cluster -- this
tends to produce more coherent, semantically-driven topics than classic bag-of-words LDA,
especially on short texts like abstracts.

`min_topic_size` is set relative to corpus size (small corpora need smaller minimum cluster
sizes, or every point ends up an "outlier" topic -1).

**Fallback:** if BERTopic can't be imported/fit (e.g. a constrained environment), we fall back
to scikit-learn's `LatentDirichletAllocation` on a bag-of-words representation -- classic LDA,
exactly the second option named in this week's brief.


In [3]:
topic_backend = None
topic_ids: List[int] = []
topic_keywords: Dict[int, List[str]] = {}

MIN_TOPIC_SIZE = max(2, len(chunks_df) // 25)

try:
    from bertopic import BERTopic

    from sklearn.feature_extraction.text import CountVectorizer as _CV
    topic_model = BERTopic(min_topic_size=MIN_TOPIC_SIZE, verbose=False, calculate_probabilities=False,
                            vectorizer_model=_CV(stop_words="english", min_df=1, ngram_range=(1, 2)))
    raw_topics, _ = topic_model.fit_transform(chunks_df["text"].tolist(), embeddings=embeddings)
    topic_ids = list(raw_topics)
    topic_backend = f"BERTopic (min_topic_size={MIN_TOPIC_SIZE})"

    for tid in set(topic_ids):
        if tid == -1:
            topic_keywords[tid] = ["outlier"]
        else:
            topic_keywords[tid] = [w for w, _ in topic_model.get_topic(tid)[:6]]

except Exception as e:
    print(f"[warn] BERTopic failed ({type(e).__name__}: {e}). Falling back to sklearn LDA.")
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.decomposition import LatentDirichletAllocation

    N_TOPICS = max(2, min(8, len(chunks_df) // 10))
    cv = CountVectorizer(max_features=5000, stop_words="english", min_df=1)
    bow = cv.fit_transform(chunks_df["text"].tolist())
    lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42, max_iter=20)
    doc_topic = lda.fit_transform(bow)
    topic_ids = doc_topic.argmax(axis=1).tolist()
    topic_backend = f"sklearn LatentDirichletAllocation (n_topics={N_TOPICS})"

    vocab = np.array(cv.get_feature_names_out())
    for tid in range(N_TOPICS):
        top_idx = lda.components_[tid].argsort()[::-1][:6]
        topic_keywords[tid] = vocab[top_idx].tolist()

chunks_df["topic_id"] = topic_ids
chunks_df["topic_label"] = chunks_df["topic_id"].map(lambda t: ", ".join(topic_keywords[t][:3]))

print(f"Topic backend: {topic_backend}")
print(f"Discovered {len(topic_keywords)} topics across {len(chunks_df):,} chunks.\n")
topic_info = (chunks_df.groupby(["topic_id", "topic_label"]).size()
              .reset_index(name="n_chunks").sort_values("n_chunks", ascending=False))
topic_info


Topic backend: BERTopic (min_topic_size=3)
Discovered 5 topics across 75 chunks.



,topic_id,topic_label,n_chunks
0,0,"allow, allow model, architecture natural",15
1,1,"network, accuracy standard, achieving",15
2,2,"analyzes, analyzes convergence, batch",15
3,3,"bm25 opendomain, bm25, biencoder trained",15
4,4,"training, cluttered environments, cluttered",15


---
## 4. Manual Validation & Edge Cases

**Manual validation:** for each discovered topic, we sample a couple of chunk titles so a human
can eyeball whether the automatically-extracted keywords actually match what those documents
are about -- this is the "validate topic outputs manually" deliverable. We also cross-tabulate
`topic_id` against the corpus's own `category` field (when present) as a sanity check: if
topics roughly align with known ArXiv categories, that's independent evidence the clustering
is picking up real structure, not noise.

**Edge cases handled:**
- **Short documents** — a chunk with too little text to support a meaningful topic vector is
  flagged (`is_short_document`) rather than silently trusted; short-document topic assignments
  are called out separately in validation rather than treated the same as a full abstract.
- **Jargon / domain-specific vocabulary** — standard English stopword removal is kept
  deliberately *narrow* (only common English stopwords, not a broader technical-term filter),
  so domain terms like "self-attention" or "hyperparameter" survive into the topic keywords
  instead of being stripped out as noise -- for a technical corpus, jargon *is* the signal.


In [4]:
MIN_CHARS_FOR_TOPIC = 40
chunks_df["is_short_document"] = chunks_df["char_count"] < MIN_CHARS_FOR_TOPIC
n_short = int(chunks_df["is_short_document"].sum())
print(f"Short-document edge case: {n_short} / {len(chunks_df)} chunks are under "
      f"{MIN_CHARS_FOR_TOPIC} characters (topic assignments for these should be treated as "
      f"low-confidence).")

print("\n--- Manual validation sample: 2 example chunks per topic ---")
for tid, keywords in sorted(topic_keywords.items(), key=lambda kv: kv[0]):
    sample = chunks_df[chunks_df["topic_id"] == tid].head(2)
    print(f"\nTopic {tid}  |  keywords: {', '.join(keywords)}")
    for _, row in sample.iterrows():
        print(f"   - {row['title']}")

if "category" in chunks_df.columns:
    print("\n--- Topic vs. corpus category cross-tab (sanity check) ---")
    crosstab = pd.crosstab(chunks_df["topic_id"], chunks_df["category"])
    display(crosstab)


Short-document edge case: 0 / 75 chunks are under 40 characters (topic assignments for these should be treated as low-confidence).

--- Manual validation sample: 2 example chunks per topic ---

Topic 0  |  keywords: allow, allow model, architecture natural, capture longrange, capture, evaluated
   - Transformer Language Model — Study 1
   - Transformer Language Model — Study 2

Topic 1  |  keywords: network, accuracy standard, achieving, achieving strong, accuracy, classification network
   - Convolutional Image Classification — Study 1
   - Convolutional Image Classification — Study 2

Topic 2  |  keywords: analyzes, analyzes convergence, batch, batch size, bounds, bounds function
   - Gradient Based Optimization — Study 1
   - Gradient Based Optimization — Study 2

Topic 3  |  keywords: bm25 opendomain, bm25, biencoder trained, biencoder, dense retrieval, dense
   - Dense Retrieval Semantic Search — Study 1
   - Dense Retrieval Semantic Search — Study 2

Topic 4  |  keywords: trainin

category,cs.CL,cs.CV,cs.IR,cs.LG,cs.RO
topic_id,,,,,
0,15,0,0,0,0
1,0,15,0,0,0
2,0,0,0,15,0
3,0,0,15,0,0
4,0,0,0,0,15


---
## 5. Named Entity Recognition

**Why NER over sentiment analysis:** this corpus is ArXiv research abstracts -- stylistically
neutral, third-person, formal writing. Sentiment analysis on that text mostly just measures
"positive-sounding results language" (e.g. "improves," "outperforms") rather than anything a
retrieval system could usefully filter on. NER, by contrast, extracts exactly the kind of
structured metadata a research corpus benefits from -- model names, methods, and datasets --
which becomes genuinely useful filterable metadata in Section 7.

**Hybrid extractor, to handle jargon:** general-purpose spaCy NER (`en_core_web_sm`) is trained
on news/Wikipedia text, so it tags known model names as generic `ORG`/`PRODUCT` entities but
misses domain terms it's never seen (e.g. "self-attention," "batch normalization"). We pair it
with a small curated ML/CS term list matched case-insensitively, so common jargon is still
captured even when spaCy's general model has no label for it. This combination is the "handle
edge cases (jargon)" requirement applied to NER specifically.


In [5]:
import spacy

nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])

DOMAIN_TERMS = [
    "transformer", "self-attention", "attention mechanism", "bert", "gpt", "resnet",
    "convolutional neural network", "cnn", "recurrent neural network", "rnn", "lstm",
    "gradient descent", "stochastic gradient descent", "sgd", "batch normalization",
    "dropout", "reinforcement learning", "policy gradient", "embedding", "fine-tuning",
    "pretraining", "cross-entropy", "backpropagation", "dense retrieval", "bi-encoder",
    "contrastive loss", "bm25", "hyperparameter", "overfitting", "regularization",
]
_DOMAIN_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(t) for t in DOMAIN_TERMS) + r")\b", re.IGNORECASE
)


def extract_entities(text: str) -> List[str]:
    '''Hybrid NER: spaCy general entities + curated domain-term matches, de-duplicated.'''
    if not text or not text.strip():
        return []
    doc = nlp(text)
    spacy_ents = [ent.text for ent in doc.ents if ent.label_ in
                  {"ORG", "PRODUCT", "PERSON", "GPE", "WORK_OF_ART", "NORP"}]
    domain_ents = [m.group(1) for m in _DOMAIN_PATTERN.finditer(text)]
    combined = spacy_ents + domain_ents
    seen, deduped = set(), []
    for e in combined:
        key = e.lower().strip()
        if key and key not in seen:
            seen.add(key)
            deduped.append(e.strip())
    return deduped


# Smoke test
print(extract_entities(
    "Transformer models like BERT use self-attention, while ResNet applies convolutional "
    "neural networks with batch normalization."
))


['BERT', 'ResNet', 'Transformer', 'self-attention', 'batch normalization']


---
## 6. Evaluate NER Accuracy Against a Small Hand-Labeled Set

A held-out gold-standard set of 10 example sentences, representative of the corpus's own
subject matter, is hand-labeled with the entities each sentence *should* yield. `extract_entities()`
is run on each sentence and scored against the gold labels with **case-insensitive, substring-aware
matching** (a predicted entity counts as a match if it or the gold label contains the other --
e.g. predicting "self-attention" against a gold label of "self attention mechanism" should
still count, since it's the same underlying concept, not a miss). Precision, recall, and F1 are
reported per-sentence and in aggregate.


In [6]:
GOLD_SET = [
    ("Transformers use self-attention to model long-range dependencies between tokens.",
     ["transformer", "self-attention"]),
    ("BERT and GPT are both built on the transformer architecture.",
     ["BERT", "GPT", "transformer"]),
    ("ResNet introduced residual connections to train very deep convolutional neural networks.",
     ["ResNet", "convolutional neural network"]),
    ("Stochastic gradient descent is the standard optimizer for training deep neural networks.",
     ["stochastic gradient descent"]),
    ("Dropout and batch normalization are common regularization techniques.",
     ["dropout", "batch normalization", "regularization"]),
    ("The bi-encoder was trained with a contrastive loss for dense retrieval.",
     ["bi-encoder", "contrastive loss", "dense retrieval"]),
    ("BM25 remains a strong sparse retrieval baseline compared to neural rankers.",
     ["BM25"]),
    ("Reinforcement learning policies were fine-tuned using policy gradient methods.",
     ["reinforcement learning", "fine-tuned", "policy gradient"]),
    ("LSTM networks were the dominant architecture for sequence modeling before transformers.",
     ["LSTM", "transformer"]),
    ("The model was pretrained on a large corpus before fine-tuning on the downstream task.",
     ["pretrained", "fine-tuning"]),
]


def fuzzy_match(pred: str, gold: str) -> bool:
    p, g = pred.lower().strip(), gold.lower().strip()
    return p == g or p in g or g in p


def score_sentence(predicted: List[str], gold: List[str]) -> dict:
    matched_gold = set()
    tp = 0
    for p in predicted:
        for i, g in enumerate(gold):
            if i not in matched_gold and fuzzy_match(p, g):
                matched_gold.add(i)
                tp += 1
                break
    precision = tp / len(predicted) if predicted else 1.0
    recall = tp / len(gold) if gold else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"tp": tp, "precision": precision, "recall": recall, "f1": f1}


eval_rows = []
for sentence, gold in GOLD_SET:
    predicted = extract_entities(sentence)
    scores = score_sentence(predicted, gold)
    eval_rows.append({"sentence": sentence[:60] + "...", "predicted": predicted,
                       "gold": gold, **scores})

ner_eval_df = pd.DataFrame(eval_rows)
print(f"NER evaluation on {len(GOLD_SET)} hand-labeled sentences:")
print(f"  Mean precision: {ner_eval_df['precision'].mean():.2f}")
print(f"  Mean recall:    {ner_eval_df['recall'].mean():.2f}")
print(f"  Mean F1:        {ner_eval_df['f1'].mean():.2f}")
ner_eval_df[["sentence", "predicted", "gold", "precision", "recall", "f1"]]


NER evaluation on 10 hand-labeled sentences:
  Mean precision: 1.00
  Mean recall:    0.77
  Mean F1:        0.85


,sentence,predicted,gold,precision,recall,f1
0,Transformers use self-attention to model long-...,[self-attention],"[transformer, self-attention]",1.0,0.500000,0.666667
1,BERT and GPT are both built on the transformer...,"[BERT, GPT, transformer]","[BERT, GPT, transformer]",1.0,1.000000,1.000000
2,ResNet introduced residual connections to trai...,[ResNet],"[ResNet, convolutional neural network]",1.0,0.500000,0.666667
3,Stochastic gradient descent is the standard op...,[Stochastic gradient descent],[stochastic gradient descent],1.0,1.000000,1.000000
4,Dropout and batch normalization are common reg...,"[Dropout, batch normalization, regularization]","[dropout, batch normalization, regularization]",1.0,1.000000,1.000000
5,The bi-encoder was trained with a contrastive ...,"[bi-encoder, contrastive loss, dense retrieval]","[bi-encoder, contrastive loss, dense retrieval]",1.0,1.000000,1.000000
6,BM25 remains a strong sparse retrieval baselin...,[BM25],[BM25],1.0,1.000000,1.000000
7,Reinforcement learning policies were fine-tune...,"[Reinforcement learning, policy gradient]","[reinforcement learning, fine-tuned, policy gr...",1.0,0.666667,0.800000
8,LSTM networks were the dominant architecture f...,[LSTM],"[LSTM, transformer]",1.0,0.500000,0.666667
9,The model was pretrained on a large corpus bef...,[fine-tuning],"[pretrained, fine-tuning]",1.0,0.500000,0.666667


---
## 7. Apply NER to the Full Corpus & Integrate NLP Metadata into ChromaDB

Every chunk gets its entities extracted and its topic assignment carried in. We then push
`topic_id`, `topic_label`, `entities` (comma-joined, since ChromaDB metadata values must be
primitives, not lists), and `n_entities` into the **existing** ChromaDB records with
`collection.update()` -- this updates metadata on already-ingested vectors *without* needing to
resupply their embeddings, so Week 2's index doesn't need to be rebuilt.

**Edge cases handled:**
- **ChromaDB update batch limits** — updates are chunked into batches of 500, same as the
  Week 2 ingestion pattern.
- **List-typed metadata rejected by ChromaDB** — entity lists are serialized to a single
  comma-separated string before being written.
- **Missing/short-document chunks** — chunks with zero extracted entities still get
  `n_entities = 0` written explicitly, rather than being skipped, so they remain queryable and
  distinguishable from chunks that were never processed.


In [7]:
tqdm.pandas(desc="Extracting entities")
chunks_df["entities"] = chunks_df["text"].progress_apply(extract_entities)
chunks_df["entities_str"] = chunks_df["entities"].apply(lambda es: ", ".join(es))
chunks_df["n_entities"] = chunks_df["entities"].apply(len)

print(f"Entity extraction complete. Mean entities/chunk: {chunks_df['n_entities'].mean():.2f} | "
      f"chunks with 0 entities: {(chunks_df['n_entities'] == 0).sum()}")

update_ids = chunks_df["chunk_id"].tolist()
update_metadatas = [
    {"topic_id": int(row.topic_id), "topic_label": row.topic_label,
     "entities": row.entities_str, "n_entities": int(row.n_entities),
     "is_short_document": bool(row.is_short_document)}
    for row in chunks_df.itertuples(index=False)
]

for start in tqdm(range(0, len(update_ids), 500), desc="Updating ChromaDB metadata"):
    end = start + 500
    try:
        collection.update(ids=update_ids[start:end], metadatas=update_metadatas[start:end])
    except Exception as e:
        print(f"[error] Metadata update batch {start}:{end} failed: {e}")

chunks_df.to_parquet(DATA_DIR / "chunks_enriched.parquet", index=False)
print(f"\nNLP metadata written to ChromaDB and saved to data/chunks_enriched.parquet.")
sample = collection.get(ids=update_ids[:1])
print("Example enriched record metadata:", sample["metadatas"][0])


Extracting entities:   0%|          | 0/75 [00:00<?, ?it/s]

Entity extraction complete. Mean entities/chunk: 2.00 | chunks with 0 entities: 0


Updating ChromaDB metadata:   0%|          | 0/1 [00:00<?, ?it/s]


NLP metadata written to ChromaDB and saved to data/chunks_enriched.parquet.
Example enriched record metadata: {'chunk_index': 0, 'topic_label': 'allow, allow model, architecture natural', 'is_short_document': False, 'entities': 'transformer, Self-attention', 'title': 'Transformer Language Model — Study 1', 'category': 'cs.CL', 'doc_id': 'synthetic.cs.CL.0000', 'published': '2024-01-15T00:00:00Z', 'topic_id': 0, 'n_entities': 2}


---
## 8. Advanced Retrieval — Filter by Topic and Category

With `topic_id` now stored as ChromaDB metadata, `filtered_search()` extends Week 2's plain
semantic search with **combined metadata filters** (`$and` over topic and/or category), so a
query can be scoped to a specific discovered topic and/or ArXiv category instead of searching
the whole corpus.


In [8]:
def filtered_search(query: str, k: int = 5, topic_id: Optional[int] = None,
                     category: Optional[str] = None) -> pd.DataFrame:
    empty = pd.DataFrame(columns=["rank", "score", "doc_id", "title", "topic_label", "entities", "text"])
    if not query.strip() or collection.count() == 0:
        return empty

    conditions = []
    if topic_id is not None:
        conditions.append({"topic_id": topic_id})
    if category is not None:
        conditions.append({"category": category})
    where = {"$and": conditions} if len(conditions) > 1 else (conditions[0] if conditions else None)

    query_embedding = np.asarray(embed_texts([query]))[0].tolist()
    results = collection.query(query_embeddings=[query_embedding], n_results=min(k, collection.count()),
                                where=where, include=["documents", "metadatas", "distances"])

    rows = []
    for rank, (doc, meta, dist) in enumerate(
            zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1):
        rows.append({"rank": rank, "score": round(1 - dist, 4), "doc_id": meta.get("doc_id", ""),
                      "title": meta.get("title", ""), "topic_label": meta.get("topic_label", ""),
                      "entities": meta.get("entities", ""), "text": doc[:150] + "..."})
    return pd.DataFrame(rows)


# Demo: unfiltered vs. topic-filtered search for the same query
demo_topic = topic_info.iloc[0]["topic_id"]
print(f"Filtering demo to topic_id={demo_topic} ('{topic_info.iloc[0]['topic_label']}')\n")
filtered_search("neural network model for a machine learning task", k=5, topic_id=int(demo_topic))


Filtering demo to topic_id=0 ('allow, allow model, architecture natural')



,rank,score,doc_id,title,topic_label,entities,text
0,1,0.3791,synthetic.cs.CL.0002,Transformer Language Model — Study 3,"allow, allow model, architecture natural","transformer, Self-attention",This paper introduces a transformer-based arch...
1,2,0.3791,synthetic.cs.CL.0003,Transformer Language Model — Study 4,"allow, allow model, architecture natural","transformer, Self-attention",This paper introduces a transformer-based arch...
2,3,0.3791,synthetic.cs.CL.0004,Transformer Language Model — Study 5,"allow, allow model, architecture natural","transformer, Self-attention",This paper introduces a transformer-based arch...
3,4,0.3791,synthetic.cs.CL.0009,Transformer Language Model — Study 10,"allow, allow model, architecture natural","transformer, Self-attention",This paper introduces a transformer-based arch...
4,5,0.3791,synthetic.cs.CL.0014,Transformer Language Model — Study 15,"allow, allow model, architecture natural","transformer, Self-attention",This paper introduces a transformer-based arch...


---
## 9. Documentation — Effectiveness & Accuracy of the Extracted NLP Metadata

### Topic modeling
**Method:** BERTopic on the Week 2 embeddings (UMAP + HDBSCAN + c-TF-IDF keywords), with
`min_topic_size` scaled to corpus size; falls back to sklearn LDA on bag-of-words if BERTopic
is unavailable. **Effectiveness:** validated two ways -- manual inspection of sample titles per
topic (Section 4), and a topic-vs-category cross-tab, which shows discovered topics
concentrating within specific ArXiv categories rather than spreading evenly across all of
them -- evidence the clusters reflect real semantic structure rather than arbitrary groupings.
**Known limitation:** on a very small or highly repetitive corpus (as in the offline fallback
mode), BERTopic can under-segment into fewer, broader topics than a large real corpus would
yield; this is expected and improves automatically as corpus size grows.

### Named entity recognition
**Method:** hybrid extractor -- spaCy's general-purpose NER for named entities (model names,
organizations, etc.) plus a curated domain-term matcher for ML/CS jargon spaCy's general model
was never trained to recognize. **Accuracy:** measured against a 10-sentence hand-labeled gold
set using fuzzy (substring-aware) matching -- see Section 6's printed precision/recall/F1 for
the exact numbers from this run. **Known limitation:** the domain-term list is curated and
finite, so genuinely novel jargon outside that list (and outside spaCy's training data) will
still be missed; extending the term list, or fine-tuning a small NER model on labeled ArXiv
abstracts, is the natural next step for higher recall on more advanced technical vocabulary.

### Edge cases handled
| Edge case | Where handled | How |
|---|---|---|
| Short documents (topic modeling) | Section 4 | Flagged via `is_short_document`, called out separately in validation rather than silently trusted |
| Jargon (topic modeling) | Section 3-4 | Narrow stopword list only -- domain terms are kept as topic keywords, not filtered out |
| Jargon (NER) | Section 5 | Curated domain-term matcher supplements spaCy's general model |
| Empty / whitespace-only text (NER) | Section 5 | `extract_entities()` returns `[]` immediately, never raises |
| List-typed values rejected by ChromaDB | Section 7 | Entity lists serialized to a comma-joined string before writing metadata |
| ChromaDB batch update limits | Section 7 | Metadata updates chunked into batches of 500 |
| Corpus/collection out of sync | Section 2 | Chunk count vs. collection count checked at load time; re-ingests if mismatched |

### Integration into retrieval
`topic_id`, `topic_label`, `entities`, and `n_entities` are now queryable ChromaDB metadata
fields. `filtered_search()` (Section 8) demonstrates combining a topic filter with the existing
category filter from Week 2 -- retrieval is no longer just "most similar chunk," it can be
"most similar chunk *within* a specific discovered theme," which is the advanced filtering
capability this week's deliverable asks for.
